# Bayesian Blocks Validation

This notebook validates `foretools.aux.bb_bins.BayesianBlocks` against `astropy.stats.bayesian_blocks` for the three supported fitness modes:

- `events`
- `regular_events`
- `measures`

The implementation under test lives in `foretools/aux/bb_bins.py`.

In [ ]:
import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.stats import bayesian_blocks

from foretools.aux.bb_bins import BayesianBlocks

print(Path('foretools/aux/bb_bins.py').resolve())
print('BayesianBlocks loaded from module:', BayesianBlocks)

In [ ]:
def compare_case(name, t, x=None, sigma=None, **kwargs):
    ours = BayesianBlocks(**kwargs).fit_edges(t, x, sigma)
    ref = bayesian_blocks(t, x=x, sigma=sigma, **kwargs)
    print(name)
    print('  ours:', ours)
    print('  ref :', ref)
    print('  match:', np.allclose(ours, ref, atol=1e-10, rtol=1e-10))
    print()
    return ours, ref


rng = np.random.default_rng(42)

events_t = rng.choice(np.arange(20), size=80, replace=True).astype(float)
events_edges, _ = compare_case('events', events_t, fitness='events')

regular_t = np.arange(0, 20, 1.0)
regular_x = (rng.random(size=regular_t.size) > 0.6).astype(int)
regular_edges, regular_ref = compare_case(
    'regular_events', regular_t, x=regular_x, fitness='regular_events', dt=1.0
)

measures_t = np.sort(rng.uniform(0, 10, size=40))
measures_x = np.sin(measures_t) + 0.1 * rng.normal(size=measures_t.size)
measures_sigma = 0.2 + 0.05 * rng.random(size=measures_t.size)
measures_edges, _ = compare_case(
    'measures',
    measures_t,
    x=measures_x,
    sigma=measures_sigma,
    fitness='measures',
)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.step(regular_t, regular_x, where='mid', label='regular event counts', color='tab:blue')

for edge in regular_edges:
    ax.axvline(edge, color='tab:green', linestyle='--', alpha=0.8)

for edge in regular_ref:
    ax.axvline(edge, color='tab:red', linestyle=':', alpha=0.6)

ax.set_title('Regular Events: our edges (green dashed) vs astropy (red dotted)')
ax.set_xlabel('time')
ax.set_ylabel('event')
ax.legend()
plt.show()

## Notes

- `events` and `measures` match the reference implementation exactly for the validation cases above.
- `regular_events` now also matches after aligning the log-likelihood handling with the `astropy` implementation.
- The corresponding automated checks live in `tests/test_bb_bins.py`.